In [1]:
 cd /kaggle/input/datasets/pavasshastri/llm-v1/llm_1/tokeniser

/kaggle/input/datasets/pavasshastri/llm-v1/llm_1/tokeniser


In [2]:
import regex as re
from tqdm.notebook import tqdm 

def encoder(utf_lists, enc_map):
  updated_lists = []
  count=0
  for utf_list in utf_lists:
    new_list = []
    if len(utf_list)<=1:
      updated_lists.append(utf_list)
      continue
    maps = [enc_map.get((utf_list[i], utf_list[i+1])) for i in range(len(utf_list)-1) if enc_map.get((utf_list[i], utf_list[i+1]))]
    if len(maps)==0:
      updated_lists.append(utf_list)
      continue
    mini = min(maps)
    i=0
    while i<len(utf_list):
      if i==len(utf_list)-1:
        new_list.append(utf_list[i])
        break
      if enc_map.get((utf_list[i], utf_list[i+1]))==mini:
        new_list.append(mini)
        i+=2
      else:
        new_list.append(utf_list[i])
        i+=1
    updated_lists.append(new_list)
    count+=1

  if count==0:
    return updated_lists

  return encoder(updated_lists, enc_map)
# def encode_text(text, enc_map):
#   if len(enc_map)==0:
#       return list(map(int, text.encode("utf-8"))) 
#   gpt2_pat =  re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
#   split_text = re.findall(gpt2_pat, text)
#   #print(f"split_text : {split_text}")
#   utf_lists = [list(map(int, t.encode("utf-8"))) for t in split_text]
#   #print(f"utf_lists : {utf_lists}")
#   encoded_lists = encoder(utf_lists, enc_map)
#   tokens = []
#   for l in encoded_lists:
#     tokens+=l
#   return tokens

def encode_text(text, enc_map):
    tokens = []
    utf_lists = [list(map(int, text.encode("utf-8")))]
    encoded_lists = encoder(utf_lists, enc_map)
    for l in encoded_lists:
        tokens+=l
    return tokens

def compute_pair_count(utf_lists, pair_map):
    maxi=0
    for utf_list in utf_lists:
        n = len(utf_list)
        i=0
        #print(f"before while loop, i : {i}, n: {n}")
        while i<n-1:
          #print(f"inside while loop...")
          if (utf_list[i], utf_list[i+1]) in pair_map:
            pair_map[(utf_list[i], utf_list[i+1])]+=1
          else:
            pair_map.update({(utf_list[i], utf_list[i+1]):1})
          maxi = max(maxi, pair_map[(utf_list[i], utf_list[i+1])])
          i+=1
    return pair_map



# def merge_pairs(utf_lists, pair_map, maxi):
#     updated_lists = []
#     for utf_list in utf_lists:
#         i=0
#         n = len(utf_list)
#         new_list = []
#         while i<n:
#           if i==n-1:
#             new_list.append(utf_list[i])
#             break
#           if pair_map[(utf_list[i], utf_list[i+1])]==maxi:
#             #print(f"entry : {(utf_list[i], utf_list[i+1])} pair_map : {pair_map[(utf_list[i], utf_list[i+1])]}")
#             if (utf_list[i], utf_list[i+1]) not in enc_map:
#               enc_map[(utf_list[i], utf_list[i+1])]=start
#               new_list.append(start)
#               start+=1
#               if start>=vocab_size:
#                   print(f"in loop exit because start>=vocab_size, start : {start}, vocab_size : {vocab_size}")
#                   return enc_map
#             else:
#               new_list.append(enc_map[(utf_list[i], utf_list[i+1])])
#             i+=2
#           else:
#             new_list.append(utf_list[i])
#             i+=1
#         updated_lists.append(new_list)
      
# [k for k in updated_lists if len(k)>1]
    
def bpe(utf_lists, vocab_size, start = 256):
  enc_map = {}
  with tqdm(desc="Training Progress", unit="items") as pbar:
      while start<vocab_size:
          #print(f"vocab learned : {start}, utf_list : {len(utf_lists)}")
          updated_lists = {}
          maxi=0
          pair_map = {}
          for utf_list in utf_lists:
            utf_list = list(utf_list)
            n = len(utf_list)
            i=0
            #print(f"before while loop, i : {i}, n: {n}")
            while i<n-1:
              #print(f"inside while loop...")
              if (utf_list[i], utf_list[i+1]) in pair_map:
                pair_map[(utf_list[i], utf_list[i+1])]+=utf_lists[tuple(utf_list)]
              else:
                pair_map.update({(utf_list[i], utf_list[i+1]):utf_lists[tuple(utf_list)]})
              maxi = max(maxi, pair_map[(utf_list[i], utf_list[i+1])])
              i+=1
              pair_count = True
              
            # if not pair_count:
            #     print(f"exit because pair_count=0, utf_list : {utf_lists[:100]}")
            #     return enc_map
          
          for utf_list in utf_lists:
            utf_list = list(utf_list)
            i=0
            n = len(utf_list)
            new_list = []
            while i<n:
              if i==n-1:
                new_list.append(utf_list[i])
                break
              if pair_map[(utf_list[i], utf_list[i+1])]==maxi:
                #print(f"entry : {(utf_list[i], utf_list[i+1])} pair_map : {pair_map[(utf_list[i], utf_list[i+1])]}")
                if (utf_list[i], utf_list[i+1]) not in enc_map:
                  enc_map[(utf_list[i], utf_list[i+1])]=start
                  new_list.append(start)
                  start+=1
                  if start>=vocab_size:
                      #print(f"in loop exit because start>=vocab_size, start : {start}, vocab_size : {vocab_size}")
                      return enc_map
                else:
                  new_list.append(enc_map[(utf_list[i], utf_list[i+1])])
                i+=2
              else:
                new_list.append(utf_list[i])
                i+=1
            if len(new_list)>1:
                updated_lists.update({tuple(new_list):utf_lists[tuple(utf_list)]})
          
          utf_lists = updated_lists
          pbar.update(1)
      
  #updated_lists = [j for j in updated_lists if len(j)>1]  
  return enc_map#bpe(updated_lists, vocab_size, start=start, enc_map=enc_map)


def train_tokeniser(text, vocab_size, special_tokens=['<|endoftext|>']):
    print("Trainer initiated...")
    spe_enc_map = {special_tokens[i]:i+1+vocab_size for i in range(len(special_tokens))}
    gpt2_pat =  re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
    print("GPT pat declared...")
    special_pattern = "(" + "|".join(re.escape(tok) for tok in special_tokens) + ")"
    print("Special pattern declared, starting splitting....")  
    parts = re.split(special_pattern, text)
    print("Splitting completed.....")
    parts = [p for p in parts if p]
    parts = [p for p in parts if p not in special_tokens]
    print("Parts formed....Computing word freq")
    word_map = {}
    for part in tqdm(parts):
        for i in re.findall(gpt2_pat, part):
            if i in word_map:
                word_map[i]+=1
            else:
                word_map.update({i:1})
    utf_lists = {tuple(list(map(int, key.encode("utf-8")))):value for key, value in word_map.items() if len(tuple(list(map(int, key.encode("utf-8")))))>1}
    print("utf_lists formed...")
    # batch_size=1
    # batched_parts = [' '.join(parts[i:min(i+batch_size, len(parts))]) for i in range(0,len(parts),batch_size)]
    # enc_map = {}
    # start = 256
    # un_words = set()
    # while start<vocab_size:
    #     pair_map={}
    #     start_prev = start
    #     for word in tqdm(word_map):
    #         #print(f"part encountered : {part}")
    #         # if countery==100000:
    #         #     break
    #         split_text = []
    #         if part not in special_tokens:
    #             split_text.extend([re.findall(gpt2_pat, part) if i not in un_words)
    #         else:
    #             split_text.append(part)
    #         un_words = set(split_text))    
    #         utf_lists = [encode_text(t, enc_map) for t in split_text if t not in special_tokens]
    #         utf_lists = [j for j in utf_lists if len(j)>1]
    #         pair_map = compute_pair_count(utf_lists, pair_map)
    #     #print(f"pair_map : {pair_map}")
    #     maxi=0
    #     for key in pair_map:
    #         maxi = max(maxi, pair_map[key])
    #     for key in pair_map:
    #         if pair_map[key]==maxi:
    #             if key not in enc_map:
    #                 enc_map.update({key:start})
    #                 start+=1

    #     print(f"vocab learned : {start}, enc_map_sample : {list(enc_map.items())[start_prev-256:start-256]},len: {len(enc_map)}")
    
    
    
    # #split_text = re.findall(gpt2_pat, text)
    # print(f"split text ready...with total splits : {len(split_text)}")
    # utf_lists = [list(map(int, t.encode("utf-8"))) for t in tqdm(split_text) if t not in special_tokens]
    # utf_lists = [j for j in utf_lists if len(j)>1]
    print(f"Formed UTF lists of {len(utf_lists)}, Starting Training.....")
    enc_map = bpe(utf_lists, vocab_size)
    spe_enc_map = {special_tokens[i]:i+1+vocab_size for i in range(len(special_tokens))}
    return enc_map, spe_enc_map

In [3]:
#from train import train_tokeniser
import pickle
from pathlib import Path

text = Path("../data/TinyStories-train.txt").read_text(encoding="utf-8")
# chunk_size = 4 * 1024 * 1024  # 4 MB
# enc_map = {}
# spe_enc_map = {}
# with Path("../data/TinyStories-train.txt").open("r", encoding="utf-8") as f:
#     while True:
#         chunk = f.read(chunk_size)
#         if not chunk:
#             break

#         # Process this chunk
#         #process(chunk)
#         enc_map_temp, spe_enc_map_temp = train_tokeniser(chunk, 16384)
#         enc_map.update(enc_map_temp)
#         spe_enc_map.update(spe_enc_map_temp)
enc_map, spe_enc_map = train_tokeniser(text, 16384)

with open("./trained/iter_1_main.pkl", "wb") as file:
    pickle.dump(enc_map, file)

with open("./trained/iter_1_main.pkl", "wb") as file:
    pickle.dump(spe_enc_map, file)

Trainer initiated...
GPT pat declared...
Special pattern declared, starting splitting....
Splitting completed.....
Parts formed....Computing word freq


  0%|          | 0/2119719 [00:00<?, ?it/s]

utf_lists formed...
Formed UTF lists of 68233, Starting Training.....


Training Progress: 0items [00:00, ?items/s]

FileNotFoundError: [Errno 2] No such file or directory: './trained/iter_1_main.pkl'

In [9]:
mkdir /kaggle/input/datasets/pavasshastri/llm-v1/llm_1/tokeniser/trained/

mkdir: cannot create directory ‘/kaggle/input/datasets/pavasshastri/llm-v1/llm_1/tokeniser/trained/’: Read-only file system


In [8]:
len(enc_map)+256

16384

In [12]:
spe_enc_map

{'<|endoftext|>': 16385}

In [13]:
with open("/kaggle/working/iter_1_spe.pkl", "wb") as file:
    pickle.dump(spe_enc_map, file)

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')